# Прокси для $\Delta V$: NII разных сроков и будущая ликвидация книги

Исследуем методы из `RISK_METRIC_PROXIES.md`, не подбирая коэффициенты по целевой $\Delta V$:

1. **Взвешенный NII по прогнозным горизонтам.** Отдельно считаем грубую сетку прогнозов и ежегодные прогнозы; сравниваем с прямым Monte Carlo дисконтированного NII.
2. **Выплаты до $H$ плюс MtM оставшейся книги.** Варьируем $H=1,2,3,5,10,20,40$ лет и смотрим, насколько $\Delta W_H$ приближается к целевой $\Delta V$.
3. Дополнительная локальная проверка: капитализация маржи с поправками на рост, дисконт и возврат ставки к среднему. Это грубая аналитическая прокси, не глобальная формула.

Основной прогнозный план — **сохранять исходную долю fixed в новых выдачах**, одинаковую до и после шока. Его построение не использует выученную политику. Вторая серия использует обученную обратную связь как диагностический ориентир: она помогает отличить ошибку замены будущей стоимости на MtM от ошибки простого плана выдач.

Рассматриваем **мгновенный** шок $r\mapsto r+100$ б.п.; средние под $P,Q$ остаются прежними. Траектории симулируются под $P$, старые и конечные договоры оцениваются по принятой в среде формуле MtM под $Q$ с депозитами по номиналу. Горизонты и прогнозные узлы не калибруются на $\Delta V$.

Операционные расходы можно оставить и согласованно учесть в потоках. Для первого чистого эксперимента рекомендуем **$\bar\gamma=0$, $\omega=0$**, сохранив $\alpha=0.9$, $\chi=5$ и остальные параметры исходной модели. Нулевая риск-аверсия здесь важнее исключения расходов: тогда математическое ожидание дисконтированных выплат соответствует критерию целевого решателя.

Обучение из папки `risk_sensitive` (нужна обновлённая версия `sweep_stationary.py` с `--operating-cost`):

```bash
python sweep_stationary.py \
  --template-solution runs/stationary_rate_growth_refined/stationary_solution.pt \
  --gammas 0 --etas 0.01 --operating-cost 0 \
  --device cuda --iterations 1500 --residual-tolerance 0.005 \
  --output runs/proxy_target_gamma0_no_opex
```

Все веса исходного чекпоинта используются только как старт; среда и задача обучения новые. Отдельного обучения в этом ноутбуке нет. По умолчанию `RUN_DIR=None`: ноутбук ищет gamma=0, eta=.01, chi=5 в готовой сетке `stationary_gamma_eta_chi5`, а при наличии отдельной версии без расходов предпочитает её. Поэтому дополнительное обучение не обязательно. При выборе модели с расходами они учитываются в потоках автоматически. Проверьте напечатанные параметры, `converged` и невязки. Даже при малой невязке выход сети — численный ориентир, а не сертифицированное истинное оптимальное значение.

In [ ]:
from pathlib import Path
from dataclasses import replace
import hashlib, json, os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

RUN_DIR = None  # auto: gamma=0, eta=.01, chi=5; предпочтительно omega=0
# Можно задать явно: 'runs/stationary_gamma_eta_chi5/g0_e0.01' 
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float64
SMOKE = bool(os.environ.get('IRRBB_NOTEBOOK_SMOKE'))
HORIZONS = (1, 2) if SMOKE else (1, 2, 3, 5, 10, 20, 40)
NII_FORECAST_KNOTS = (1, 2) if SMOKE else (1, 2, 3, 5, 10, 20, 40)
PATHS = 8 if SMOKE else 128  # можно увеличить до 512/1024 на GPU
SEED = 2049
SHOCK = .01
PLANS = ('hold_mix', 'learned')  # learned — диагностический ориентир
RECOMPUTE_SIMULATIONS = False

root=next((q for q in (Path.cwd(),*Path.cwd().parents)
           if (q/'continuous_time_irrbb/risk_sensitive/stationary_solver.py').exists()),None)
if root is None:raise FileNotFoundError('Запустите Jupyter внутри проекта IRRBB')
sys.path.insert(0,str(root))
from continuous_time_irrbb.risk_sensitive.stationary_solver import StationarySolution
from continuous_time_irrbb.risk_sensitive.environment import Environment
def select_proxy_run(risk_root):
    if RUN_DIR is not None:
        selected=Path(RUN_DIR).expanduser()
        return selected if selected.is_absolute() else risk_root/selected
    candidates=[]
    for group in ('proxy_target_gamma0_no_opex','stationary_gamma_eta_chi5'):
        parent=risk_root/'runs'/group
        for report_path in sorted(parent.rglob('report.json')):
            if not report_path.with_name('stationary_solution.pt').exists():continue
            item=json.loads(report_path.read_text())
            params=item.get('parameters',{})
            config=item.get('stationary_config',{})
            if not (np.isclose(config.get('risk_aversion_per_loan',-1),0)
                    and np.isclose(params.get('floating_demand_slope',-1),.01)
                    and np.isclose(params.get('growth_rate_sensitivity',0),5)):
                continue
            diagnostics=item.get('diagnostics',[])
            residual=diagnostics[-1].get('bellman_rmse',float('inf')) if diagnostics else float('inf')
            candidates.append((params.get('operating_cost',.01)!=0,
                               not item.get('converged_on_validation_cloud',False),
                               residual,str(report_path.parent)))
    if not candidates:
        raise FileNotFoundError(
            'Не найден чекпоинт gamma=0, eta=.01, chi=5 в '
            'runs/proxy_target_gamma0_no_opex или runs/stationary_gamma_eta_chi5. '
            'Укажите RUN_DIR явно. Нулевые операционные расходы не обязательны: '
            'можно использовать gamma=0 из готовой сетки chi5.')
    return Path(sorted(candidates)[0][-1])

run_dir=select_proxy_run(root/'continuous_time_irrbb/risk_sensitive')
checkpoint=run_dir/'stationary_solution.pt'
if not checkpoint.exists():
    raise FileNotFoundError(f'{checkpoint}: проверьте RUN_DIR и имя подпапки модели')
print('Чекпоинт эксперимента:',checkpoint)
solution=StationarySolution.load(checkpoint,device=DEVICE,dtype=DTYPE)
env,p,cfg=solution.environment,solution.environment.p,solution.config
report=json.loads((run_dir/'report.json').read_text())
if DEVICE=='cpu':torch.set_num_threads(min(torch.get_num_threads(),4))
if not p.signed_owner_flow:raise ValueError('Эксперимент предполагает знаковые выплаты владельцу')
if not 0<p.dividend_fraction<=1:raise ValueError('Нужно alpha>0 для сравнения потоков владельцу')
if sorted(set(HORIZONS))!=list(HORIZONS) or min(HORIZONS)<1:raise ValueError('Горизонты должны возрастать')
if sorted(set(NII_FORECAST_KNOTS))!=list(NII_FORECAST_KNOTS):raise ValueError('Узлы должны возрастать')
if not set(HORIZONS).issubset(NII_FORECAST_KNOTS):raise ValueError('Каждый H должен быть прогнозным узлом')
if any(int(h)!=h for h in (*HORIZONS,*NII_FORECAST_KNOTS)):raise ValueError('Используйте целые годы')
MAX_H=max(HORIZONS)
if max(NII_FORECAST_KNOTS)!=MAX_H:raise ValueError('Последний узел должен совпадать с последним H')
if not np.isclose(round(1/cfg.dt)*cfg.dt,1):raise ValueError('Шаг модели должен делить один год')
steps_per_year=round(1/cfg.dt)
plt.style.use('seaborn-v0_8-whitegrid')
last=report['diagnostics'][-1]
print(f'{run_dir.name}: gamma_bar={cfg.risk_aversion_per_loan:g}; omega={p.operating_cost:g}; '
      f'alpha={p.dividend_fraction:g}; chi={p.growth_rate_sensitivity:g}; dt={cfg.dt:g}; device={DEVICE}')
print(f"converged={report.get('converged_on_validation_cloud')}; Bellman RMSE={last['bellman_rmse']:.6f}")
if cfg.risk_aversion_per_loan!=0:
    print('ВНИМАНИЕ: целевой V риск-чувствителен. Сравнение с ожидаемыми потоками включает риск-поправку; '
          'проверка Bellman-согласованности через обычное ожидание для него не применяется.')
if p.operating_cost!=0:
    print('Расходы включены: основной сопоставимый прогнозный поток — alpha*profit. '
          'Отдельно сохранён и показан gross-NII без вычитания расходов.')

## Одни и те же состояния для всех прокси

Все сценарии — срезы при текущем $L=L_0$. Ставки старых fixed-договоров и спреды float зафиксированы одинаково. Для каждого сценария обе ветви шока получают одинаковые случайные приращения; одинаковые приращения используются и между планами. Стандартная ошибка оценивается по **парным разностям**. Она отражает Monte Carlo-шум, но не ошибку целевой сети, временной схемы или прокси.

Для `hold_mix` фиксируем $f=F_0/L_0$ на всех будущих выдачах, в обеих ветвях. Это прогнозный план без адаптации к шоку; новые кредиты всё равно выдаются, фондирование и книга меняются по исходным правилам среды. Для `learned` каждое действие определяется текущим состоянием своей ветви.

In [ ]:
r0,F0,Y0,E0=map(float,report['reference_state'])
L0=p.loans
old_float=p.floating_spread_0-p.floating_demand_slope*(1-F0/L0)
old_fixed=(Y0-old_float*(L0-F0))/F0 if F0>1e-8 else float(
    env.new_fixed_coupon(torch.tensor(r0,device=DEVICE,dtype=DTYPE)))
specs=[('Исходная книга',r0,F0/L0,E0/L0),('Низкая ставка',.04,.5,.12),
       ('Высокая ставка',.12,.5,.12),('20% fixed',r0,.2,.12),
       ('80% fixed',r0,.8,.12),('Пониженный капитал',r0,.5,.105),
       ('Вся книга float',r0,0.,.12),('Вся книга fixed',r0,1.,.12)]
if SMOKE:specs=specs[:3]
states=torch.tensor([(r,L0*a,old_fixed*L0*a+old_float*L0*(1-a),L0*e)
                     for _,r,a,e in specs],device=DEVICE,dtype=DTYPE)
bump=torch.tensor([SHOCK,0,0,0],device=DEVICE,dtype=DTYPE)
labels=[s[0] for s in specs];N_SC=len(labels)
with torch.no_grad():
    target_base=solution.value(states).cpu().numpy()
    target_delta=(solution.value(states+bump)-solution.value(states)).cpu().numpy()
    mtm_base=env.terminal_value(states).cpu().numpy()
    mtm_delta=(env.terminal_value(states+bump)-env.terminal_value(states)).cpu().numpy()
    frozen_delta=SHOCK*(states[:,3]-states[:,1]+(1-p.deposit_beta)*p.deposits).cpu().numpy()
print('Сценарий                             V до    ΔV цели   ΔMtM     ΔNII frozen1')
for name,v,dv,dm,dn in zip(labels,target_base,target_delta,mtm_delta,frozen_delta):
    print(f'{name:32s} {v:9.3f} {dv:+9.3f} {dm:+9.3f} {dn:+9.3f}')

## Метод 1: взвешенные прогнозы NII

Для накопленного прогноза $N_H^\pi=\mathbb E^P\int_0^H n_t^\pi dt$ используем
$$\widehat{PVNII}_H^\pi=\sum_{H_i\le H}w_i(r)(N_{H_i}^\pi-N_{H_{i-1}}^\pi),\qquad
w_i(r)=\frac1{H_i-H_{i-1}}\int_{H_{i-1}}^{H_i}P_P(r,t)dt.$$
После шока пересчитываются **и** доходы, **и** веса. Доходы разных интервалов не считаются дважды. $P_P$ — ожидаемый дисконт под $P$, в CIR он вычисляется аналитически. Равномерность доходов внутри интервалов и произведение средних вместо среднего произведения — допущения прокси.

Чтобы сопоставить этот прогноз со стоимостью владельца, основной график использует $\alpha\widehat{PV\Pi}$: те же веса применены к прогнозам прибыли после операционных расходов. При $\omega=0$ это ровно $\alpha\widehat{PVNII}$. Gross-NII также сохраняется. Сравнение ежегодных узлов и редкой сетки показывает влияние усреднения по интервалам; сравнение с прямым Monte Carlo $PV\Pi$ — совокупное влияние ковариации потока с дисконтом и временной аппроксимации.

## Метод 2: выплаты до H плюс MtM остатка

$$W_H^\pi=\mathbb E^P\left[\int_0^H B_t\alpha\Pi_t^\pi dt+B_H MtM(X_H^\pi,L_H)\right].$$
При $H=0$ это MtM. Вторая часть оценивает **оставшиеся** договоры и капитал; предшествующие выплаты — только реально распределённая часть прибыли. Поэтому удержанная прибыль не считается дважды. Поток всей NII без коэффициента выплаты вместе с конечным капиталом такой интерпретации не имеет.

$W_H$ соответствует условному прекращению деятельности в H. Оно не является гарантированной нижней границей V: ликвидация отсутствует среди действий исходной задачи, и правила оценки под P/Q различаются. При интегрируемом потоке и исчезающей дисконтированной конечной стоимости $W_H\to J_0^\pi$ при $H\to\infty$. Монотонность ошибки по H не гарантируется; 5 лет могут оставлять большую часть бессрочного потока в хвосте.

In [ ]:
CACHE_DIR=(Path('/private/tmp/irrbb_value_proxy_smoke') if SMOKE
           else run_dir/'value_proxy_cache')
@torch.no_grad()
def policy_actions(x,loans,plan):
    if plan=='hold_mix':
        share=states[:,1]/L0
        return share[:,None,None].expand(N_SC,2,PATHS).reshape(-1)
    if plan!='learned':raise ValueError(plan)
    return torch.cat([solution.action(part,loans=ll,action_count=cfg.action_count)
                      for part,ll in zip(x.split(128),loans.split(128))])

@torch.no_grad()
def simulate_plan(plan):
    identity={'version':1,'checkpoint':hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
              'states':states.cpu().tolist(),'plan':plan,'paths':PATHS,'seed':SEED,
              'max_horizon':MAX_H,'shock':SHOCK,'dt':cfg.dt,'dtype':str(DTYPE),
              'action_count':cfg.action_count}
    key=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:20]
    cache=CACHE_DIR/f'{plan}_{key}.npz'
    if cache.exists() and not RECOMPUTE_SIMULATIONS:
        print('кеш:',cache)
        with np.load(cache) as data:return {k:data[k].copy() for k in data.files}
    x=states[:,None,None,:].expand(N_SC,2,PATHS,4).clone()
    x[:,1,:,0]+=SHOCK
    x=x.reshape(-1,4)
    loans=torch.full(x.shape[:-1],L0,device=DEVICE,dtype=DTYPE)
    discount=torch.ones_like(loans)
    nii=torch.zeros_like(loans);profit=torch.zeros_like(loans)
    pv_nii=torch.zeros_like(loans);pv_owner=torch.zeros_like(loans)
    result={k:[] for k in ('nii','profit','pv_nii','owner','terminal_mtm','terminal_value')}
    def record():
        terminal_mtm=discount*env.terminal_value(x,loans=loans)
        terminal_value=torch.cat([solution.value(part,loans=ll)
                                 for part,ll in zip(x.split(256),loans.split(256))])*discount
        for name,arr in (('nii',nii),('profit',profit),('pv_nii',pv_nii),
                         ('owner',pv_owner),('terminal_mtm',terminal_mtm),('terminal_value',terminal_value)):
            if not torch.isfinite(arr).all():raise ValueError(f'Non-finite {plan}/{name}; inspect paths')
            result[name].append(arr.reshape(N_SC,2,PATHS).cpu().numpy().copy())
    record()
    gen=torch.Generator(device='cpu').manual_seed(SEED)
    for step_index in range(MAX_H*steps_per_year):
        # Common rate innovations across base/shock, scenarios and forecast plans.
        noise=torch.randn(PATHS,generator=gen,dtype=DTYPE).to(DEVICE)
        noise=noise[None,None,:].expand(N_SC,2,PATHS).reshape(-1)
        f=policy_actions(x,loans,plan)
        coupon=env.new_fixed_coupon(x[:,0])
        transition=env.step(x,f,noise,cfg.dt,time=step_index*cfg.dt,
                            loans=loans,validate=False,fixed_coupon=coupon)
        half_decay=np.exp(-p.amortization*cfg.dt/2)
        mid_issuance=transition.midpoint_loans-half_decay*loans
        margin=f*coupon+(1-f)*env.new_floating_spread(1-f)
        mid=torch.stack((x[:,0],half_decay*x[:,1]+mid_issuance*f,
                         half_decay*x[:,2]+mid_issuance*margin,transition.midpoint_equity),dim=-1)
        mid_profit=env.profit(mid,time=(step_index+.5)*cfg.dt,loans=transition.midpoint_loans)
        mid_nii=mid_profit+p.operating_cost*(p.deposits/p.loans)*transition.midpoint_loans
        torch.testing.assert_close(transition.dividend_rate,p.dividend_fraction*mid_profit,
                                   rtol=1e-8,atol=1e-8)
        nii=nii+mid_nii*cfg.dt
        profit=profit+mid_profit*cfg.dt
        pv_nii=pv_nii+discount*mid_nii*env.exp_integral(-x[:,0],cfg.dt)
        pv_owner=pv_owner+discount*transition.dividend_pv
        discount=discount*transition.discount
        x,loans=transition.state,transition.next_loans
        if (step_index+1)%steps_per_year==0:
            record()
            year=(step_index+1)//steps_per_year
            if year%5==0 or year==MAX_H:print(f'{plan}: {year}/{MAX_H} лет',flush=True)
    arrays={key:np.stack(value) for key,value in result.items()}
    CACHE_DIR.mkdir(parents=True,exist_ok=True)
    temporary=cache.with_suffix('.tmp')
    with temporary.open('wb') as stream:np.savez_compressed(stream,**arrays)
    temporary.replace(cache)
    return arrays

simulations={plan:simulate_plan(plan) for plan in PLANS}
for plan,data in simulations.items():
    # H=0 liquidation must reproduce the analytical instantaneous MtM shock.
    zero=data['terminal_mtm'][0,:,1]-data['terminal_mtm'][0,:,0]
    np.testing.assert_allclose(zero,mtm_delta[:,None]*np.ones((1,PATHS)),atol=1e-8)
print('Симуляции готовы; исходные политики и чекпоинты не изменены.')

In [ ]:
# Expected discount under P, without changing loan pricing in simulation.
discount_env=Environment(replace(p,kappa_q=p.kappa_p,mean_rate_q=p.mean_rate_p)).to(device=DEVICE,dtype=DTYPE)
nodes,weights=np.polynomial.legendre.leggauss(32)
nodes=torch.as_tensor(nodes,device=DEVICE,dtype=DTYPE)
weights=torch.as_tensor(weights,device=DEVICE,dtype=DTYPE)
initial_rates=torch.stack((states[:,0],states[:,0]+SHOCK),dim=-1)

@torch.no_grad()
def interval_weights(knots):
    output=[];previous=0
    for horizon in knots:
        t=previous+(nodes+1)*(horizon-previous)/2
        prices=discount_env.bond_q(initial_rates[...,None],t)
        # average discount on interval; duration cancels in the mapping from GL nodes.
        output.append((prices*weights).sum(-1).cpu().numpy()/2)
        previous=horizon
    return np.stack(output)

def weighted_forecasts(cumulative,knots):
    cumulative=cumulative[np.r_[0,knots]]
    increments=np.diff(cumulative,axis=0)
    return np.cumsum(increments*interval_weights(knots)[...,None],axis=0)

def paired(array):
    # All arrays: [horizon, scenario, branch, path]. No independent-branch SE.
    difference=array[:,:,1]-array[:,:,0]
    return difference.mean(-1),difference.std(-1,ddof=1)/np.sqrt(PATHS)

estimates={}
knots=np.array(NII_FORECAST_KNOTS,dtype=int)
horizons=np.array(HORIZONS,dtype=int)
coarse_ids=np.array([list(knots).index(h) for h in horizons])
for plan,data in simulations.items():
    coarse_gross=p.dividend_fraction*weighted_forecasts(data['nii'],knots)[coarse_ids]
    coarse_net=p.dividend_fraction*weighted_forecasts(data['profit'],knots)[coarse_ids]
    annual_net=p.dividend_fraction*weighted_forecasts(data['profit'],np.arange(1,MAX_H+1))[horizons-1]
    owner=data['owner'][horizons]
    liquidation=data['owner'][horizons]+data['terminal_mtm'][horizons]
    consistency=data['owner'][horizons]+data['terminal_value'][horizons]
    estimates[plan]={'coarse':coarse_net,'coarse_gross':coarse_gross,'annual':annual_net,
                     'owner':owner,'liquidation':liquidation,'consistency':consistency}
    # Without operating expenses, alpha*discounted NII equals actual owner payout PV.
    if p.operating_cost==0:
        np.testing.assert_allclose(data['owner'],p.dividend_fraction*data['pv_nii'],rtol=1e-8,atol=1e-8)
    print('\nПлан:',plan)
    print('H    Δweighted_NII(net)      ΔW_H        SE(ΔW_H)      ΔV цель')
    for i,h in enumerate(horizons):
        wn,_=paired(coarse_net);wh,se=paired(liquidation)
        print(f'{h:2d}   {wn[i,0]:+12.4f}    {wh[i,0]:+12.4f}   {se[i,0]:9.4f}   {target_delta[0]:+9.4f}')

## Горизонт и ошибка прокси

Сплошные линии сравнивают разные приближения в исходной книге. Полосы у $W_H$ — ±1.96 стандартной ошибки **парной** Monte Carlo-разности; это не полный доверительный интервал ошибки метода. Горизонтальная линия $\Delta\widehat V$ сама является аппроксимацией решателя.

`hold_mix` — самостоятельная прокси по простому плану. `learned` использует сохранённую оптимизированную обратную связь и поэтому служит диагностикой. Их нельзя выдавать за два одинаково независимых способа прогнозировать V. В обоих случаях $W_0=MtM$, но при H>0 меняются и выдача, и состояние книги.

In [ ]:
fig,axes=plt.subplots(1,len(PLANS),figsize=(8*len(PLANS),5.5),squeeze=False,constrained_layout=True)
for ax,plan in zip(axes.flat,PLANS):
    result=estimates[plan]
    for key,label,color,style in (
        ('coarse','Взвешенные NII-прогнозы (net)','C0','-'),
        ('annual','То же, ежегодные прогнозы','C0','--'),
        ('owner','Прямой PV выплат до H','C2',':'),
        ('liquidation','Выплаты + конечный MtM','C1','-')):
        mean,se=paired(result[key])
        xx=np.r_[0,horizons] if key=='liquidation' else horizons
        yy=np.r_[mtm_delta[0],mean[:,0]] if key=='liquidation' else mean[:,0]
        ax.plot(xx,yy,marker='o',label=label,color=color,ls=style)
        if key=='liquidation':
            ax.fill_between(horizons,mean[:,0]-1.96*se[:,0],mean[:,0]+1.96*se[:,0],color=color,alpha=.15)
    ax.axhline(target_delta[0],color='black',lw=1.7,label='ΔV сохранённой сети')
    ax.axhline(mtm_delta[0],color='gray',ls=':',label='ΔMtM текущей книги')
    ax.set(title=plan,xlabel='горизонт H, годы',ylabel='изменение стоимости, ден. ед.')
    ax.legend(fontsize=8)
fig.suptitle('Исходная книга: разные прокси при одном мгновенном шоке +100 б.п.')
plt.show()

fig,axes=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
for plan in PLANS:
    for key,style in (('coarse','--'),('liquidation','-')):
        mean,_=paired(estimates[plan][key])
        rmse=np.sqrt(np.mean((mean-target_delta[None,:])**2,axis=1))
        axes[0].plot(horizons,rmse,marker='o',ls=style,label=f'{plan}: {key}')
    base=estimates[plan]['liquidation'][:,0,0].mean(-1)
    mean,_=paired(estimates[plan]['liquidation'])
    relative=np.divide(100*mean[:,0],base,out=np.full_like(base,np.nan),where=base>.01*L0)
    axes[1].plot(horizons,relative,marker='o',label=f'ΔW_H/W_H: {plan}')
axes[0].set(title='Ошибка по всем заданным состояниям',xlabel='H, годы',ylabel='RMSE относительно ΔV сети, ден. ед.')
axes[1].axhline(100*target_delta[0]/target_base[0],color='black',label='ΔV/V исходной книги')
axes[1].set(title='Относительная дельта в исходной книге',xlabel='H, годы',ylabel='% от собственной стоимости до шока')
for ax in axes:ax.legend(fontsize=8)
plt.show()

In [ ]:
# Scenario-by-horizon comparison: same monetary scale for all panels.
panels=[]
for plan in PLANS:
    for key,title in (('coarse','Взвешенный NII(net)'),('liquidation','Выплаты + MtM')):
        mean,_=paired(estimates[plan][key]);panels.append((f'{plan}: {title}',mean.T))
panels.append(('ΔV сети',target_delta[:,None]))
limit=max(.001,max(float(np.abs(data).max()) for _,data in panels))
fig,axes=plt.subplots(1,len(panels),figsize=(4.5*len(panels),6),sharey=True,
                       squeeze=False,constrained_layout=True)
for ax,(title,data) in zip(axes.flat,panels):
    im=ax.imshow(data,aspect='auto',cmap='RdBu_r',vmin=-limit,vmax=limit)
    ax.set(title=title,yticks=range(N_SC),yticklabels=labels,
           xticks=range(data.shape[1]),xticklabels=[str(h) for h in horizons] if data.shape[1]>1 else ['∞'])
    ax.set_xlabel('H, годы')
fig.colorbar(im,ax=list(axes.flat),label='денежные единицы; одна цветовая шкала')
plt.show()

## Что объясняет остаточную ошибку?

Проверка ниже применяется только при $\bar\gamma=0$. Для сохранённой политики строим
$$C_H=\mathbb E^P\left[\int_0^H B_t\alpha\Pi_t^{\widehat\pi}dt+B_H\widehat V(X_H,L_H)\right].$$
При точном риск-нейтральном решении и согласованном переходе это совпало бы с V. Расхождение $\Delta C_H-\Delta\widehat V$ показывает накопленную несогласованность сети/её политики и численной интеграции. Monte Carlo использует нормальные приращения, а решатель — конечную гауссову квадратуру; это ещё один источник расхождения.

При одинаковых действиях разность $\Delta W_H-\Delta C_H$ — ровно изменение ошибки замены продолжения $\widehat V$ на конечное MtM. Она включает стоимость бизнеса после H, правила оценки P/Q и принятую оценку фондирования. Если $\Delta C_H$ само заметно отличается от $\Delta\widehat V$, нельзя приписывать всю ошибку W плохой прокси.

Для weighted-NII отдельно показываем разность редкой и ежегодной сеток (интерполяция прогнозов), а затем ежегодной сетки и прямого дисконтирования потока (ковариация и временная аппроксимация). Здесь не требуется, чтобы эти ошибки имели определённый знак.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
for plan in PLANS:
    c,_=paired(estimates[plan]['coarse']);a,_=paired(estimates[plan]['annual'])
    direct,_=paired(estimates[plan]['owner'])
    axes[0].plot(horizons,c[:,0]-a[:,0],marker='o',label=f'{plan}: редкая − ежегодная')
    axes[0].plot(horizons,a[:,0]-direct[:,0],marker='x',ls='--',label=f'{plan}: ежегодная − прямой PV')
axes[0].axhline(0,color='black',lw=.7)
axes[0].set(title='Источники ошибки weighted-NII, исходная книга',xlabel='H, годы',ylabel='денежные единицы')
axes[0].legend(fontsize=8)
if cfg.risk_aversion_per_loan==0 and 'learned' in estimates:
    check,check_se=paired(estimates['learned']['consistency'])
    wh,_=paired(estimates['learned']['liquidation'])
    axes[1].plot(horizons,check[:,0],marker='o',label='ΔC_H: выплаты + продолжение сети')
    axes[1].fill_between(horizons,check[:,0]-1.96*check_se[:,0],check[:,0]+1.96*check_se[:,0],alpha=.15)
    axes[1].plot(horizons,wh[:,0],marker='o',label='ΔW_H: выплаты + MtM')
    axes[1].axhline(target_delta[0],color='black',label='ΔV сети')
    axes[1].set(title='Отделяем ошибку продолжения от качества цели',xlabel='H, годы',ylabel='денежные единицы')
    axes[1].legend(fontsize=8)
else:
    axes[1].axis('off');axes[1].text(.05,.5,'Для проверки продолжения нужны gamma=0 и план learned.',transform=axes[1].transAxes)
plt.show()
if not report.get('converged_on_validation_cloud',False):
    print('ВНИМАНИЕ: целевая модель не достигла своего критерия сходимости. '
          'Ошибки прокси измерены относительно предварительной ΔV, не истинного оптимума.')

## Дополнительно: локальная капитализация маржи

Для грубой локальной модели с постоянными удельной выплатой $m=\alpha\Pi/L$, ростом g и дисконтом r стоимость потока равна $Lm/(r-g)$, если $r>g$. Если начальный шок ставки затухает как $\delta e^{-\kappa_P t}$, её линеаризованная чувствительность имеет вид
$$\Delta W_{\rm local}\approx\frac{L\delta}{r-g+\kappa_P}
\left[m_r+\frac{m(g_r-1)}{r-g}\right],$$
где $m_r=\alpha[(L-F)+C-\beta D]/L$ при замороженной структуре, $g_r=-\chi(\mu+g(r))$.

В скобках видны изменения удельной маржи, роста и дисконта. Это приближение использует текущие значения коэффициентов как локальную опорную траекторию; оно игнорирует изменение структуры, капитала и политики. Для состояний с $r\le g(r)$ выводим пропуск, а не формально бесконечное значение: в полной mean-reverting-модели стоимость при этом всё ещё может быть конечной.

In [ ]:
with torch.no_grad():
    rr,ff,yy,ee=states.unbind(-1)
    growth=env.growth_rate(rr)
    margin=p.dividend_fraction*env.profit(states)/L0
    margin_r=p.dividend_fraction*(L0-ff+env.cash(states)-p.deposit_beta*p.deposits)/L0
    growth_r=-p.growth_rate_sensitivity*(p.amortization+growth)
    gap=rr-growth
    valid=gap>1e-6
    local=torch.full_like(rr,float('nan'))
    local[valid]=L0*SHOCK/(gap[valid]+p.kappa_p)*(margin_r[valid]+
                  margin[valid]*(growth_r[valid]-1)/gap[valid])
    local=local.cpu().numpy()
fig,ax=plt.subplots(figsize=(12,5),constrained_layout=True)
pos=np.arange(N_SC);width=.36
ax.bar(pos-width/2,target_delta,width,label='ΔV целевой сети')
ax.bar(pos+width/2,local,width,label='Локальная прокси маржа + рост + дисконт')
ax.set(xticks=pos,xticklabels=labels,ylabel='денежные единицы',title='Локальная аналитическая проверка')
ax.tick_params(axis='x',rotation=25);ax.legend();ax.axhline(0,color='black',lw=.7)
plt.show()
for name,value,ok in zip(labels,local,valid.cpu().numpy()):
    print(name, f'{value:+.4f}' if ok else 'пропуск: текущая r <= g(r)')

## Как выбирать удачную прокси

Не выбираем горизонт только потому, что он попал в одну точку ΔV. Смотрим на ошибку по всем сценариям, её изменение с H и устойчивость при увеличении PATHS/смене SEED. Хорошее совпадение относительных дельт может объясняться разными знаменателями — проверяем денежные дельты и уровни стоимости отдельно.

Главная проверка независимости — результат простого `hold_mix`. Если хорошо работает только `learned`, это аргумент в пользу конечной оценки продолжения после управления, но не дешёвая независимая прокси оптимального решения. Если сетка прогнозов NII груба, полезно сначала уплотнить `NII_FORECAST_KNOTS`, не меняя уже сохранённые симуляции. Последний горизонт при этом должен оставаться тем же.

Эмпирическое обучение смеси NII/MtM здесь пока не реализовано: сначала проверяем экономически определённые конструкции. Исходные чекпоинты и правила среды не меняются.